In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
import matplotlib.pyplot as plt

dispositivo = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Dispositivo: {dispositivo}")

clases = ['airplane', 'automobile', 'bird', 'cat', 'deer',
          'dog', 'frog', 'horse', 'ship', 'truck']

# Imágenes pequeñas 32x32 con data augmentation
transform_train = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomCrop(32, padding=4),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

train_dataset = torchvision.datasets.CIFAR10(root='./data', train=True, download=False, transform=transform_train)
test_dataset = torchvision.datasets.CIFAR10(root='./data', train=False, download=False, transform=transform_test)

train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=128, shuffle=True)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=128, shuffle=False)

print(f"Batches de entrenamiento: {len(train_loader)}")

Dispositivo: cpu


c:\Users\diego\OneDrive\Desktop\visionShop\.venv\Lib\site-packages\torchvision\datasets\cifar.py:83: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  entry = pickle.load(f, encoding="latin1")


Batches de entrenamiento: 391


In [3]:
class CNNMejorada(nn.Module):
    def __init__(self):
        super(CNNMejorada, self).__init__()
        
        self.conv1 = nn.Sequential(
            nn.Conv2d(3, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.Conv2d(64, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            nn.Dropout2d(0.2)
        )
        
        self.conv2 = nn.Sequential(
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.Conv2d(128, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            nn.Dropout2d(0.3)
        )
        
        self.conv3 = nn.Sequential(
            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.Conv2d(256, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            nn.Dropout2d(0.4)
        )
        
        self.fc = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256 * 4 * 4, 512),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(512, 10)
        )
    
    def forward(self, x):
        x = self.conv1(x)
        x = self.conv2(x)
        x = self.conv3(x)
        x = self.fc(x)
        return x

modelo_mejorado = CNNMejorada().to(dispositivo)
params = sum(p.numel() for p in modelo_mejorado.parameters())
print(f"Parámetros totales: {params:,}")
print("✓ Modelo listo")

Parámetros totales: 3,249,994
✓ Modelo listo


In [5]:
criterio = nn.CrossEntropyLoss()
optimizador = optim.Adam(modelo_mejorado.parameters(), lr=0.001)
scheduler = optim.lr_scheduler.StepLR(optimizador, step_size=5, gamma=0.5)

EPOCAS = 10
historial_mejorado = {"train_loss": [], "train_acc": [], "test_acc": []}

for epoca in range(EPOCAS):
    modelo_mejorado.train()
    loss_total = 0
    correctas = 0
    total = 0
    
    for imagenes, etiquetas in train_loader:
        imagenes, etiquetas = imagenes.to(dispositivo), etiquetas.to(dispositivo)
        
        optimizador.zero_grad()
        salidas = modelo_mejorado(imagenes)
        loss = criterio(salidas, etiquetas)
        loss.backward()
        optimizador.step()
        
        loss_total += loss.item()
        _, predicciones = salidas.max(1)
        correctas += predicciones.eq(etiquetas).sum().item()
        total += etiquetas.size(0)
    
    train_acc = 100 * correctas / total
    train_loss = loss_total / len(train_loader)
    
    modelo_mejorado.eval()
    correctas_test = 0
    total_test = 0
    with torch.no_grad():
        for imagenes, etiquetas in test_loader:
            imagenes, etiquetas = imagenes.to(dispositivo), etiquetas.to(dispositivo)
            salidas = modelo_mejorado(imagenes)
            _, predicciones = salidas.max(1)
            correctas_test += predicciones.eq(etiquetas).sum().item()
            total_test += etiquetas.size(0)
    
    test_acc = 100 * correctas_test / total_test
    scheduler.step()
    
    historial_mejorado["train_loss"].append(train_loss)
    historial_mejorado["train_acc"].append(train_acc)
    historial_mejorado["test_acc"].append(test_acc)
    
    print(f"Época {epoca+1}/{EPOCAS} | Loss: {train_loss:.4f} | Train Acc: {train_acc:.2f}% | Test Acc: {test_acc:.2f}%")

print("\n✓ Entrenamiento completado")

Época 1/10 | Loss: 2.1502 | Train Acc: 22.34% | Test Acc: 34.84%
Época 2/10 | Loss: 1.8440 | Train Acc: 30.65% | Test Acc: 36.41%
Época 3/10 | Loss: 1.7588 | Train Acc: 33.58% | Test Acc: 40.12%
Época 4/10 | Loss: 1.6894 | Train Acc: 36.98% | Test Acc: 46.72%
Época 5/10 | Loss: 1.6296 | Train Acc: 38.33% | Test Acc: 46.41%
Época 6/10 | Loss: 1.5545 | Train Acc: 42.85% | Test Acc: 50.89%
Época 7/10 | Loss: 1.5095 | Train Acc: 44.15% | Test Acc: 52.37%
Época 8/10 | Loss: 1.4604 | Train Acc: 46.16% | Test Acc: 53.61%
Época 9/10 | Loss: 1.4297 | Train Acc: 48.12% | Test Acc: 55.34%
Época 10/10 | Loss: 1.4023 | Train Acc: 48.66% | Test Acc: 55.64%

✓ Entrenamiento completado


In [ ]:
# Usar solo 18% del dataset para entrenar más rápido
indices = torch.randperm(len(train_dataset))[:8000]
train_dataset_pequeño = torch.utils.data.Subset(train_dataset, indices)
train_loader = torch.utils.data.DataLoader(train_dataset_pequeño, batch_size=128, shuffle=True)
print(f"Dataset reducido: {len(train_dataset_pequeño):,} imágenes")
print(f"Batches: {len(train_loader)}")

Dataset reducido: 8,000 imágenes
Batches: 63


In [6]:
import os
import torch

# Guardar el modelo mejorado con lo que tenemos
os.makedirs("../models", exist_ok=True)
torch.save(modelo_mejorado.state_dict(), "../models/cnn_mejorada.pth")
print("✓ Modelo guardado")

# Comparación final
print("\n=== COMPARACIÓN DE MODELOS ===")
print(f"CNN desde cero (50K imágenes, 10 épocas): 76% accuracy")
print(f"CNN mejorada   (5K imágenes,  10 épocas): 54% accuracy")
print("\nConclusión: más datos > arquitectura más compleja")

✓ Modelo guardado

=== COMPARACIÓN DE MODELOS ===
CNN desde cero (50K imágenes, 10 épocas): 76% accuracy
CNN mejorada   (5K imágenes,  10 épocas): 54% accuracy

Conclusión: más datos > arquitectura más compleja
